In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
PROJECT_ROOT = Path.cwd().parent
INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "pca_clustering_inputs"

# Load all six PCA representations
pca_candidates = {
    "Operational_2PC": pd.read_parquet(INPUT_DIR / "Operational_2PC.parquet"),
    "Operational_5PC": pd.read_parquet(INPUT_DIR / "Operational_5PC.parquet"),
    "Telemetry_4PC": pd.read_parquet(INPUT_DIR / "Telemetry_4PC.parquet"),
    "Telemetry_7PC": pd.read_parquet(INPUT_DIR / "Telemetry_7PC.parquet"),
    "Broad_6PC": pd.read_parquet(INPUT_DIR / "Broad_6PC.parquet"),
    "Broad_10PC": pd.read_parquet(INPUT_DIR / "Broad_10PC.parquet"),
}

# Load the common track index for reference
common_track_index = pd.read_parquet(
    INPUT_DIR / "common_track_index.parquet"
)["track_index"]

# Validate all representations
for name, df in pca_candidates.items():
    print(
        f"{name:20} | "
        f"shape={str(df.shape):14} | "
        f"finite={np.isfinite(df.to_numpy()).all()} | "
        f"unique_index={df.index.is_unique}"
    )

# Confirm all six use the same row index
reference_index = next(iter(pca_candidates.values())).index

all_indices_match = all(
    df.index.equals(reference_index)
    for df in pca_candidates.values()
)

print("\nAll candidate indexes match:", all_indices_match)
print("Common track index length:", len(common_track_index))
print("Candidate row count:", len(reference_index))


Operational_2PC      | shape=(42867, 2)     | finite=True | unique_index=True
Operational_5PC      | shape=(42867, 5)     | finite=True | unique_index=True
Telemetry_4PC        | shape=(42867, 4)     | finite=True | unique_index=True
Telemetry_7PC        | shape=(42867, 7)     | finite=True | unique_index=True
Broad_6PC            | shape=(42867, 6)     | finite=True | unique_index=True
Broad_10PC           | shape=(42867, 10)    | finite=True | unique_index=True

All candidate indexes match: True
Common track index length: 42867
Candidate row count: 42867


In [3]:

variables_to_check = [
    "Z_operational",
    "Z_telemetry",
    "X_10_scaled_common",
    "operational_features",
    "telemetry_features",
    "selected_features",
    "X_cluster_5",
    "X_cluster_7",
    "X_cluster_10",
    "OUTPUT_DIR",
]

for name in variables_to_check:
    print(f"{name:25} | {'Exists' if name in globals() else 'MISSING'}")


Z_operational             | MISSING
Z_telemetry               | MISSING
X_10_scaled_common        | MISSING
operational_features      | MISSING
telemetry_features        | MISSING
selected_features         | MISSING
X_cluster_5               | MISSING
X_cluster_7               | MISSING
X_cluster_10              | MISSING
OUTPUT_DIR                | MISSING


In [5]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "pca_clustering_inputs"

# Load the three original-feature matrices
original_candidates = {
    "Operational_Original": pd.read_parquet(
        INPUT_DIR / "Operational_Original.parquet"
    ),
    "Telemetry_Original": pd.read_parquet(
        INPUT_DIR / "Telemetry_Original.parquet"
    ),
    "Broad_Original": pd.read_parquet(
        INPUT_DIR / "Broad_Original.parquet"
    ),
}

# Combine original and PCA representations
candidates = {
    **original_candidates,
    **pca_candidates,
}

# Validate all nine candidates
reference_index = candidates["Operational_Original"].index

print(f"{'Representation':25} | {'Shape':15} | Finite | Index matches")
print("-" * 75)

for name, df in candidates.items():
    finite = np.isfinite(df.to_numpy()).all()
    index_match = df.index.equals(reference_index)

    print(
        f"{name:25} | {str(df.shape):15} | "
        f"{str(finite):6} | {index_match}"
    )

assert len(candidates) == 9
assert all(df.shape[0] == 42867 for df in candidates.values())
assert all(np.isfinite(df.to_numpy()).all() for df in candidates.values())
assert all(df.index.equals(reference_index) for df in candidates.values())

print("\nAll nine representations passed validation.")


Representation            | Shape           | Finite | Index matches
---------------------------------------------------------------------------
Operational_Original      | (42867, 5)      | True   | True
Telemetry_Original        | (42867, 7)      | True   | True
Broad_Original            | (42867, 10)     | True   | True
Operational_2PC           | (42867, 2)      | True   | True
Operational_5PC           | (42867, 5)      | True   | True
Telemetry_4PC             | (42867, 4)      | True   | True
Telemetry_7PC             | (42867, 7)      | True   | True
Broad_6PC                 | (42867, 6)      | True   | True
Broad_10PC                | (42867, 10)     | True   | True

All nine representations passed validation.



# Clustering Representation Design

## 1. Objective

Evaluate how feature selection and dimensionality reduction affect the discovery of operational patterns in DT-CARGO vehicle tracks.

The experiment compares original standardized features with reduced and full PCA representations using the same 42,867 tracks.

## 2. Candidate Representations

| Feature set | Original features | Reduced PCA | Full PCA |
|---|---:|---:|---:|
| Operational | 5 | 2 PCs | 5 PCs |
| Operational + telemetry | 7 | 4 PCs | 7 PCs |
| Broad telemetry | 10 | 6 PCs | 10 PCs |

The reduced representations retain at least 90% of the variance in their corresponding PCA experiments.

## 3. Hypotheses

**H1 — Original versus full PCA**

When all principal components are retained, K-Means should produce equivalent distance geometry to K-Means on the corresponding standardized original features, subject to numerical precision and implementation details.

**H2 — Reduced versus full PCA**

Removing lower-variance components may improve clustering by reducing noise or irrelevant variation. It may also remove useful distinctions. The direction of the effect is unknown.

**H3 — Operational versus telemetry representations**

Adding signal-loss and telemetry-quality features may reveal additional track patterns. However, clusters could reflect data-recording quality rather than operational behavior.

**H4 — Broad representation**

The broad feature set may capture more diverse operational and telemetry patterns, but additional features may also complicate cluster interpretation.

**H5 — Cluster quality is multidimensional**

A higher silhouette score alone does not establish a better operational segmentation. Cluster sizes, stability, runtime, and interpretability must also be considered.

## 4. Experimental Controls

- Use the same 42,867 tracks for every representation.
- Preserve the validated row ordering and feature preprocessing.
- Use the same K-Means implementation and distance metric.
- Compare the same candidate values of k.
- Control random initialization using recorded random seeds.
- Record preprocessing, parameters, runtime, and evaluation results.
- Do not use track identifiers as clustering features.
- Do not interpret clusters as fuel efficiency, mechanical faults, or driver behavior without supporting evidence.

## 5. Evaluation Criteria

1. **Silhouette score:** assess separation and cohesion; higher is generally better.
2. **Cluster-size distribution:** identify tiny or highly imbalanced clusters.
3. **Stability:** assess whether similar clusters emerge across random initializations.
4. **Runtime:** compare computational cost.
5. **Operational interpretability:** determine whether clusters represent meaningful track patterns.
6. **Telemetry-quality dependence:** assess whether clusters are primarily associated with signal loss or recording quality.

## 6. Planned Workflow

1. Run K-Means baselines across all nine representations.
2. Compare candidate values of k using consistent evaluation criteria.
3. Investigate promising candidates and test their stability.
4. Interpret clusters using track metadata and original features.
5. Select a representation based on evidence, not explained variance or silhouette alone.
